# Setup

In [1]:
# define root path of project 
from pathlib import Path
import sys

ROOT = Path("..").resolve()
sys.path.append(str(ROOT))

# load general setup
from utils.setup_general import *

Environment loaded


Los datos de cultivos de la UPRA están procesados en el panel CEDE y también en los datos originales de la UPRA. Acá exploro ambos datos para decidir cuales uso

# Explorar Panel CEDE

In [2]:
# Cargar datos
CEDE_agricultura = pd.read_stata(
    filepath_or_buffer = RAW/'aaa_panel_CEDE/Microdatos/PANEL_AGRICULTURA_Y_TIERRA(2024).dta',
    preserve_dtypes=True)

# Reparar variables 
CEDE_agricultura['CODIGO_MUNICIPIO'] = CEDE_agricultura['codmpio'].astype("Int64").astype(str).str.zfill(5)
CEDE_agricultura['ANNO'] = CEDE_agricultura['ano'].astype("Int64")

# Desfragmentar el DF
CEDE_agricultura = CEDE_agricultura.copy()

# Conservar solo las variables de interes
vbles_aguacate = CEDE_agricultura.columns[CEDE_agricultura.columns.str.contains('aguacate')].to_list()
vbles_cafe = CEDE_agricultura.columns[CEDE_agricultura.columns.str.contains('cafe')].to_list()
vbles_id = ['codmpio', 'CODIGO_MUNICIPIO', 'ANNO']
vbles_interes = vbles_id + vbles_aguacate + vbles_cafe
CEDE_agricultura = CEDE_agricultura[vbles_interes]

In [3]:
# Organizar datos en formato long
vbles_resultado = list(CEDE_agricultura.columns[CEDE_agricultura.columns.str.contains('aguacate')])
panel_agricultura = CEDE_agricultura.melt(
    id_vars=vbles_id,
    value_vars=vbles_interes,
    value_name='valor'
)

# Mostrar información disponible de cada variable de aguacate
display(panel_agricultura.groupby(['variable', 'ANNO'])['valor'].count().unstack().transpose())

variable,ac_aguacate,ac_aguacateh,ac_aguacatenhnp,ac_aguacatepl,ac_cafe,as_aguacate,as_aguacateh,as_aguacatenhnp,as_aguacatepl,as_cafe,p_aguacate,p_aguacateh,p_aguacatenhnp,p_aguacatepl,p_cafe,r_aguacate,r_aguacateh,r_aguacatenhnp,r_aguacatepl,r_cafe
ANNO,,,,,,,,,,,,,,,,,,,,
2003,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
2004,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
2005,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
2006,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
2007,199,0,0,0,591,233,0,0,0,593,199,0,0,0,591,199,0,0,0,591
2008,225,0,0,0,598,267,0,0,0,598,225,0,0,0,598,225,0,0,0,598
2009,243,0,0,0,610,287,0,0,0,614,243,0,0,0,610,243,0,0,0,610
2010,274,0,0,0,618,312,0,0,0,620,274,0,0,0,618,274,0,0,0,618
2011,307,0,0,0,625,340,0,0,0,627,307,0,0,0,625,307,0,0,0,625


# Explorar datos UPRA

## UPRA 2019-2024

In [4]:
UPRA = pd.read_excel(
    io= RAW/'aab_UPRA/20250617_BaseAgricola20192024.xlsx',
    sheet_name='BasePagina',
    skiprows=7,
    dtype={'Código Dane municipio':str, 'Año':int})
display(UPRA.head(3))

,Código Dane departamento,Departamento,Código Dane municipio,Municipio,Desagregación cultivo,Cultivo,Ciclo del cultivo,Grupo cultivo,Subgrupo,Año,Periodo,Área sembrada (ha),Área cosechada (ha),Producción (t),Rendimiento (t/ha),Nombre científico del cultivo,Código del cultivo,Estado físico del cultivo
0,5,Antioquia,05001,Medellín,Aguacate demás variedades,Aguacate,Permanente,Frutales,Demás frutales,2019,2019,24.00,23.00,138.00,6.00,Persea americana,2040299,En fresco
1,5,Antioquia,05001,Medellín,Aguacate demás variedades,Aguacate,Permanente,Frutales,Demás frutales,2020,2020,8.52,3.52,21.12,6.00,Persea americana,2040299,En fresco
2,5,Antioquia,05001,Medellín,Aguacate demás variedades,Aguacate,Permanente,Frutales,Demás frutales,2021,2021,8.52,4.52,27.12,6.00,Persea americana,2040299,En fresco


In [5]:
# Mientras procesaba los datos pensé que los demás cultivos compiten directamente por
# los factores de producción para los cultivos. Por eso voy a cargar el area sembrada
# y cosechada de los demas cultivos.

# Entendiendo el cultivo del aguacate hass y el aguacate papelillo como sustitutos
# Voy a hacer el análisis por cultivo y no por desagregación de cultivo.
# Por ejemplo, hago el análisis por "Aguacate" y no por "Aguacate Hass"
# El supuesto es válido en tanto variedades del mismo cultivo usen los
# factores de producción con intensidades similares

# Mostrar los cultivos disponibles en la base de datos

# ciclo del cultivo
print(MSC_SEPARADOR, "Ciclo\n", sorted(UPRA['Ciclo del cultivo'].unique()))
print(len(sorted(UPRA['Ciclo del cultivo'].unique())))

# Grupos de cultivos
print(MSC_SEPARADOR, "Grupo\n", sorted(UPRA['Grupo cultivo'].unique()))
print(len(sorted(UPRA['Grupo cultivo'].unique())))

# Grupos de cultivos
print(MSC_SEPARADOR, "Subgrupo\n", sorted(UPRA['Subgrupo'].unique()))
print(len(sorted(UPRA['Subgrupo'].unique())))


# Cultivos
print(MSC_SEPARADOR, "Cultivo\n", sorted(UPRA['Cultivo'].unique()))
print(len(sorted(UPRA['Cultivo'].unique())))


-------------------------------- Ciclo
 ['Permanente', 'Transitorio']
2

-------------------------------- Grupo
 ['Cereales', 'Cultivos para condimentos, bebidas medicinales y aromáticas', 'Cultivos tropicales tradicionales', 'Frutales', 'Hortalizas', 'Leguminosas', 'Oleaginosas', 'Raíces y tubérculos']
8

-------------------------------- Subgrupo
 ['Anonáceas', 'Aromáticas', 'Aráceas', 'Caducifolios', 'Cereales', 'Condimentos', 'Cultivos para condimentos, bebidas medicinales y aromáticas', 'Cultivos tropicales tradicionales', 'Cítricos', 'Demás frutales', 'Hortalizas', 'Hortalizas de flor', 'Hortalizas de fruto', 'Hortalizas de hoja', 'Hortalizas de raíz', 'Hortalizas de tallo', 'Leguminosas', 'Medicinales', 'Mirtáceas', 'Oleaginosas', 'Pasifloráceas', 'Raíces y tubérculos', 'Solanáceas']
23

-------------------------------- Cultivo
 ['Acelga', 'Achiote', 'Achira', 'Agraz - mortiño', 'Aguacate', 'Ahuyama', 'Ajo', 'Ajonjolí', 'Ají', 'Albahaca', 'Alcachofa', 'Algodón', 'Anón', 'Apio', 

## UPRA 2007-2018

In [6]:
UPRAantiguo = pd.read_excel(
    io= RAW/'aab_UPRA/Base Agrícola EVA 2007-2018_MADR.xlsx',
    sheet_name='FINAL',
    skiprows=1,
    dtype={'CÓD. MUN.':str, 'AÑO':int})
display(UPRAantiguo.head(3))

,CÓD. \nDEP.,DEPARTAMENTO,CÓD. MUN.,MUNICIPIO,GRUPO \nDE CULTIVO,SUBGRUPO \nDE CULTIVO,CULTIVO,DESAGREGACIÓN REGIONAL Y/O SISTEMA PRODUCTIVO,AÑO,PERIODO,Área Sembrada\n(ha),Área Cosechada\n(ha),Producción\n(t),Rendimiento\n(t/ha),ESTADO FISICO PRODUCCION,NOMBRE \nCIENTIFICO,CICLO DE CULTIVO,Unnamed: 17
0,15,BOYACA,15114,BUSBANZA,HORTALIZAS,ACELGA,ACELGA,ACELGA,2006,2006B,2.00,1.00,1.00,1.00,FRUTO FRESCO,BETA VULGARIS,TRANSITORIO,NaN
1,25,CUNDINAMARCA,25754,SOACHA,HORTALIZAS,ACELGA,ACELGA,ACELGA,2006,2006B,82.00,80.00,"1,440.00",18.00,FRUTO FRESCO,BETA VULGARIS,TRANSITORIO,NaN
2,25,CUNDINAMARCA,25214,COTA,HORTALIZAS,ACELGA,ACELGA,ACELGA,2006,2006B,1.50,1.50,26.00,17.33,FRUTO FRESCO,BETA VULGARIS,TRANSITORIO,NaN


In [7]:
# Mostrar los cultivos disponibles en la base de datos

# ciclo del cultivo
print(MSC_SEPARADOR, "Ciclo\n", sorted(UPRAantiguo['CICLO DE CULTIVO'].unique()))
print(len(sorted(UPRAantiguo['CICLO DE CULTIVO'].unique())))

# Grupos de cultivos
print(MSC_SEPARADOR, "Grupo\n", sorted(UPRAantiguo['GRUPO \nDE CULTIVO'].unique()))
print(len(sorted(UPRAantiguo['GRUPO \nDE CULTIVO'].unique())))

# Subgrupo
print(MSC_SEPARADOR, "Subgrupo\n", sorted(UPRAantiguo['SUBGRUPO \nDE CULTIVO'].unique()))
print(len(sorted(UPRAantiguo['SUBGRUPO \nDE CULTIVO'].unique())))

# Cultivo
print(MSC_SEPARADOR, "CULTIVO\n", sorted(UPRAantiguo['CULTIVO'].unique()))
print(len(sorted(UPRAantiguo['CULTIVO'].unique())))


-------------------------------- Ciclo
 ['ANUAL', 'PERMANENTE', 'TRANSITORIO']
3

-------------------------------- Grupo
 ['CEREALES', 'FIBRAS', 'FLORES Y FOLLAJES', 'FORESTALES', 'FRUTALES', 'HONGOS', 'HORTALIZAS', 'LEGUMINOSAS', 'OLEAGINOSAS', 'OTROS PERMANENTES', 'OTROS TRANSITORIOS', 'PLANTAS AROMATICAS, CONDIMENTARIAS Y MEDICINALES', 'TUBERCULOS Y PLATANOS']
13

-------------------------------- Subgrupo
 ['ACELGA', 'ACHIRA', 'AGUACATE', 'AHUYAMA', 'AJI', 'AJO', 'AJONJOLI', 'ALCACHOFA', 'ALFALFA', 'ALGARROBO', 'ALGODON', 'ANON', 'APIO', 'ARANDANO', 'ARRACACHA', 'ARROZ', 'ARVEJA', 'AVENA', 'BANANO', 'BATATA', 'BERENJENA', 'BORE', 'BROCOLI', 'CACAO', 'CADUCIFOLIOS', 'CAFE', 'CALABACIN', 'CALABAZA', 'CAUCHO', 'CAÑA', 'CAÑA FLECHA', 'CEBADA', 'CEBOLLA', 'CENTENO', 'CHACHAFRUTO', 'CHAMPIÑON', 'CHONQUE', 'CILANTRO', 'CITRICOS', 'COCO', 'COL', 'CURUBA', 'ESPARRAGO', 'ESPARTO', 'ESPINACA', 'ESTROPAJO', 'FEIJOA', 'FIQUE', 'FLORES', 'FLORES Y FOLLAJES', 'FOLLAJES', 'FRAMBUESA', 'FRESA', 'FR

## Armonizar taxonomías
Explorar las taxonomías de ambas bases de datos para armonizarlas

### Exportar clasificaciones para diligenciar manualmente

In [8]:
# Revisar area sembrada registrada en todos los años del Upra antiguo y de UPRA nuevo
# para crear el crosswalk y armonizar las taxonomías
# Lo hago por registro de area sembrada para poder garantizar la trazabilidad de los
# registros más importantes por area sembrada en Colombia

UPRAantiguo_crosswalk = UPRAantiguo.groupby(
    ['CICLO DE CULTIVO', 'GRUPO \nDE CULTIVO', 'CULTIVO'])['Área Sembrada\n(ha)'].sum().reset_index()
UPRAantiguo_crosswalk = UPRAantiguo_crosswalk.rename(columns={
        'CICLO DE CULTIVO': 'ciclo_original',
        'GRUPO \nDE CULTIVO': 'grupo_original',
        'CULTIVO': 'cultivo_original',
        'Área Sembrada\n(ha)': 'area_sembrada_ha'
    })
UPRAantiguo_crosswalk['año'] = "2007-2018"
UPRAantiguo_crosswalk['fuente_taxonomia'] = 'Eva antiguo'


UPRA_crosswalk = UPRA.groupby(
    ['Ciclo del cultivo', 'Grupo cultivo', 'Cultivo'])['Área sembrada (ha)'].sum().reset_index()
UPRA_crosswalk = UPRA_crosswalk.rename(columns={
        'Ciclo del cultivo': 'ciclo_original',
        'Grupo cultivo': 'grupo_original',
        'Cultivo': 'cultivo_original',
        'Área sembrada (ha)': 'area_sembrada_ha'
    })
UPRA_crosswalk['año'] = "2019-2024"
UPRA_crosswalk['fuente_taxonomia'] = 'Eva nuevo'


# Definir plantilla para diligenciar manualmente
plantilla = (
    pd.concat([UPRAantiguo_crosswalk, UPRA_crosswalk], ignore_index=True)
    .sort_values(['area_sembrada_ha'], ascending=False)
)
# Columnas para definir taxonomías armonizadas
plantilla[['ciclo_armonizado',
           'grupo_armonizado',
           'cultivo_armonizado',
          'notas']] = ""

# Exporto los datos para revisarlos manualmente y definir la armonización de las categorías
plantilla.to_excel(DATA/ 'config/e1001_crosswalk_cultivos.xlsx', index=False)

### Armonizar taxonomías

In [9]:
# Cargar clasificación que fue armonizada a mano
clasificacion_armonizada = pd.read_excel(DATA/"config/e1001_crosswalk_cultivos_armonizado.xlsx",
                                        sheet_name="Sheet1")
print(MSC_SEPARADOR, "\nArchivo de clasificación armonizada")
display(clasificacion_armonizada.head(2))

# En la armonización de la taxonomía, se logró armonizar más del 98% de los datos de area sembrada
# del EVA 2007-2018 y del EVA 2019-2024
print(MSC_SEPARADOR, "\nPorcentaje de area sembrada que se logró armonizar")
display(
    clasificacion_armonizada[['porcentaje acumulado armonizado  2018', 'porcentaje acumulado armonizado  2019']]
        .max().to_frame()*100)

# Conservar únicamente los cultivos que fueron armonizados
clasificacion_armonizada = clasificacion_armonizada[clasificacion_armonizada['Armonizado']==True]

# Idea general:
# En las columnas ['ciclo_original', grupo_original', 'cultivo_original'] están los nombres originales
# en las bases de datos EVA. 
# Los nombres armonizados quedaron en las columnas ['ciclo_armonizado',	'grupo_armonizado',	'cultivo_armonizado']
clasificacion_armonizada_2007_2018 = clasificacion_armonizada[clasificacion_armonizada['año']=='2007-2018']
clasificacion_armonizada_2019_2024  = clasificacion_armonizada[clasificacion_armonizada['año']=='2019-2024']

# Preparar DF para el merge con la lsita armonziada
UPRAantiguo = UPRAantiguo.rename(columns={'CULTIVO':'cultivo_original'})
UPRA = UPRA.rename(columns={'Cultivo':'cultivo_original'})

# en la lista armonizada, conservar solo las columnas necesarias
clasificacion_armonizada_2007_2018 = clasificacion_armonizada_2007_2018[[
    'cultivo_original','ciclo_armonizado','grupo_armonizado','cultivo_armonizado']]
clasificacion_armonizada_2019_2024 = clasificacion_armonizada_2019_2024[[
    'cultivo_original','ciclo_armonizado','grupo_armonizado','cultivo_armonizado']]


-------------------------------- 
Archivo de clasificación armonizada


,ciclo_original,grupo_original,cultivo_original,area_sembrada_ha,año,fuente_taxonomia,ciclo_armonizado,grupo_armonizado,cultivo_armonizado,notas,Armonizado,porcentaje acumulado armonizado 2018,porcentaje acumulado armonizado 2019
0,PERMANENTE,OTROS PERMANENTES,CAFE,"10,940,392.32",2007-2018,Eva antiguo,Permanente,Cultivos tropicales tradicionales,Café,NaN,True,0.18,0.00
1,TRANSITORIO,CEREALES,MAIZ,"7,606,746.69",2007-2018,Eva antiguo,Transitorio,Cereales,Maíz,NaN,True,0.31,0.00



-------------------------------- 
Porcentaje de area sembrada que se logró armonizar


,0
porcentaje acumulado armonizado 2018,98.83
porcentaje acumulado armonizado 2019,99.98


In [10]:
# Revisar duplicados por el valor en "cultivo_original"
# Cualquier duplicado debería tener los mismos valores en todas las filas
display(clasificacion_armonizada_2007_2018[clasificacion_armonizada_2007_2018.
    duplicated(keep=False, subset='cultivo_original')].sort_values(by='cultivo_original')
       )
# En la inspección de los duplicados no se encuentran problemas

display(clasificacion_armonizada_2019_2024[clasificacion_armonizada_2019_2024.
    duplicated(keep=False, subset='cultivo_original')].sort_values(by='cultivo_original')
       )
# En la lista armonizada para 2019_2024 no hay repetidos


,cultivo_original,ciclo_armonizado,grupo_armonizado,cultivo_armonizado
54,AJI,Transitorio,Hortalizas,Ají
142,AJI,Transitorio,Hortalizas,Ají
178,ALBAHACA,Permanente,"Cultivos para condimentos, bebidas medicinales...",Albahaca
138,ALBAHACA,Permanente,"Cultivos para condimentos, bebidas medicinales...",Albahaca
218,CURCUMA,Permanente,Raíces y tubérculos,Cúrcuma o azafrán
160,CURCUMA,Permanente,Raíces y tubérculos,Cúrcuma o azafrán
177,ESPARRAGO,Transitorio,Hortalizas,Espárrago
119,ESPARRAGO,Transitorio,Hortalizas,Espárrago
191,OREGANO,Permanente,"Cultivos para condimentos, bebidas medicinales...",Orégano
215,OREGANO,Permanente,"Cultivos para condimentos, bebidas medicinales...",Orégano


,cultivo_original,ciclo_armonizado,grupo_armonizado,cultivo_armonizado


In [11]:
# Eliminar duplicados
clasificacion_armonizada_2007_2018 = clasificacion_armonizada_2007_2018.drop_duplicates(subset='cultivo_original')
clasificacion_armonizada_2019_2024 = clasificacion_armonizada_2019_2024.drop_duplicates(subset='cultivo_original')

In [12]:
# Revisar suma de valores de los indicadores antes y después del merge
print(MSC_SEPARADOR, "UPRA 2007-2018: Resumen de todos los valores ANTES de armonizar")
antes = UPRAantiguo[['Área Sembrada\n(ha)',	'Área Cosechada\n(ha)',	'Producción\n(t)', 'Rendimiento\n(t/ha)']].aggregate(['sum', 'count'])
display(antes)

# Agregar columnas de datos armonizados
UPRAantiguo_armonizado = UPRAantiguo.merge(
    clasificacion_armonizada_2007_2018,
    on='cultivo_original',
    how='right', # Mantener solo los cultivos que existen en la lista armonizada
    validate='m:1'
)
print(MSC_SEPARADOR, "UPRA 2007-2018: Resumen de todos los valores DESPUES de armonizar")
despues = UPRAantiguo_armonizado[['Área Sembrada\n(ha)',	'Área Cosechada\n(ha)',	'Producción\n(t)', 'Rendimiento\n(t/ha)']].aggregate(['sum', 'count'])
display(despues)

print(MSC_SEPARADOR, "UPRA 2007-2018: Porcentaje de magnitudes que se conservan: despues / antes")
display(100*despues/antes)


-------------------------------- UPRA 2007-2018: Resumen de todos los valores ANTES de armonizar


,Área Sembrada\n(ha),Área Cosechada\n(ha),Producción\n(t),Rendimiento\n(t/ha)
sum,"60,334,863.76","51,807,216.06","573,927,302.13","1,919,060.51"
count,"210,847.00","210,847.00","210,847.00","207,289.00"



-------------------------------- UPRA 2007-2018: Resumen de todos los valores DESPUES de armonizar


,Área Sembrada\n(ha),Área Cosechada\n(ha),Producción\n(t),Rendimiento\n(t/ha)
sum,"59,627,601.62","51,524,887.31","571,988,858.77","1,893,706.02"
count,"207,389.00","207,389.00","207,389.00","204,336.00"



-------------------------------- UPRA 2007-2018: Porcentaje de magnitudes que se conservan: despues / antes


,Área Sembrada\n(ha),Área Cosechada\n(ha),Producción\n(t),Rendimiento\n(t/ha)
sum,98.83,99.46,99.66,98.68
count,98.36,98.36,98.36,98.58


In [13]:
# Revisar suma de valores de los indicadores antes y después del merge
print(MSC_SEPARADOR, "UPRA 2019-2024: Resumen de todos los valores ANTES de armonizar")
antes = UPRA[['Área sembrada (ha)',	'Área cosechada (ha)',	'Producción (t)', 'Rendimiento (t/ha)']].aggregate(['sum', 'count'])
display(antes)

# Agregar columnas de datos armonizados
UPRA_armonizado = UPRA.merge(
    clasificacion_armonizada_2019_2024,
    on='cultivo_original',
    how='right', # Mantener solo los cultivos que existen en la lista armonizada
    validate='m:1'
)
print(MSC_SEPARADOR, "UPRA 2019-2024: Resumen de todos los valores DESPUES de armonizar")
despues = UPRA_armonizado[['Área sembrada (ha)',	'Área cosechada (ha)',	'Producción (t)', 'Rendimiento (t/ha)']].aggregate(['sum', 'count'])
display(despues)

print(MSC_SEPARADOR, "UPRA 2019-2024: Porcentaje de magnitudes que se conservan: despues / antes")
display(100*despues/antes)


-------------------------------- UPRA 2019-2024: Resumen de todos los valores ANTES de armonizar


,Área sembrada (ha),Área cosechada (ha),Producción (t),Rendimiento (t/ha)
sum,"32,491,336.82","29,707,563.72","450,336,864.70","1,495,042.11"
count,"141,073.00","141,073.00","141,073.00","141,073.00"



-------------------------------- UPRA 2019-2024: Resumen de todos los valores DESPUES de armonizar


,Área sembrada (ha),Área cosechada (ha),Producción (t),Rendimiento (t/ha)
sum,"32,484,957.56","29,701,729.43","450,303,360.63","1,494,265.79"
count,"140,924.00","140,924.00","140,924.00","140,924.00"



-------------------------------- UPRA 2019-2024: Porcentaje de magnitudes que se conservan: despues / antes


,Área sembrada (ha),Área cosechada (ha),Producción (t),Rendimiento (t/ha)
sum,99.98,99.98,99.99,99.95
count,99.89,99.89,99.89,99.89


In [14]:
# Unir los dataframes armonizados de 2007_2018 con 2019_2024

# Definir nombres unificados para las columnas
COL_area_sembrada = 'area_sembrada_ha'
COL_area_cosechada = 'area_cosechada_ha'
COL_produccion = 'produccion_t'
COL_rendimiento = 'rendimiento_t_ha'

# Unificar nombres de las columnas en UPRA 2019-2024
UPRA_armonizado = UPRA_armonizado.rename(columns={
    'Código Dane municipio':COL_ID_MUNICIPIO,
    'Año': COL_ANNO,
    'Área sembrada (ha)': COL_area_sembrada,
    'Área cosechada (ha)': COL_area_cosechada,
    'Producción (t)': COL_produccion,
    'Rendimiento (t/ha)': COL_rendimiento
})

# Unificar nombres de las columnas en UPRA 2007-2018
UPRAantiguo_armonizado = UPRAantiguo_armonizado.rename(columns={
    'CÓD. MUN.':COL_ID_MUNICIPIO,
    'AÑO': COL_ANNO,
    'Área Sembrada\n(ha)': COL_area_sembrada,
    'Área Cosechada\n(ha)': COL_area_cosechada,
    'Producción\n(t)': COL_produccion,
    'Rendimiento\n(t/ha)': COL_rendimiento
})


# Definir columnas a conservar
columnas_df_final = [COL_ID_MUNICIPIO,
                    COL_ANNO,
                    COL_area_sembrada,
                    COL_area_cosechada,
                    COL_produccion,
                    COL_rendimiento,
                     'ciclo_armonizado',
                     'grupo_armonizado',
                     'cultivo_armonizado']

# Unir DF
UPRA_2007_2024 = pd.concat([
    UPRAantiguo_armonizado[columnas_df_final],
    UPRA_armonizado[columnas_df_final]
])

# Asegurarse que el tipo de cada columna es el adecuado
UPRA_2007_2024["codigo_dane_municipio"] = (
    UPRA_2007_2024["codigo_dane_municipio"]
    .astype(str)
    .str.zfill(5) # El código de municipio es un string de 5 caracteres
)
UPRA_2007_2024[COL_ANNO] = UPRA_2007_2024[COL_ANNO].astype(int) # el año es un int

columnas_numericas = [COL_area_sembrada, COL_area_cosechada, COL_produccion, COL_rendimiento]
UPRA_2007_2024[columnas_numericas] = UPRA_2007_2024[columnas_numericas].astype(float) # Las columnas numericas son floats

columnas_taxonomia_cultivos = ['ciclo_armonizado', 'grupo_armonizado', 'cultivo_armonizado']
UPRA_2007_2024[columnas_taxonomia_cultivos] = UPRA_2007_2024[columnas_taxonomia_cultivos].astype(str) # Las columnas de la taxonomía de los cultivos son strings

# Revisar la información del DF
print(MSC_SEPARADOR, " Tipo de datos en cada columna")
display(UPRA_2007_2024.info())
print(MSC_SEPARADOR, " Estadísticas descriptivas de las columnas")
display(UPRA_2007_2024.describe())


--------------------------------  Tipo de datos en cada columna
<class 'pandas.core.frame.DataFrame'>
Index: 348313 entries, 0 to 140923
Data columns (total 9 columns):
 #   Column                 Non-Null Count   Dtype  
---  ------                 --------------   -----  
 0   codigo_dane_municipio  348313 non-null  object 
 1   anno                   348313 non-null  int32  
 2   area_sembrada_ha       348313 non-null  float64
 3   area_cosechada_ha      348313 non-null  float64
 4   produccion_t           348313 non-null  float64
 5   rendimiento_t_ha       345260 non-null  float64
 6   ciclo_armonizado       348313 non-null  object 
 7   grupo_armonizado       348313 non-null  object 
 8   cultivo_armonizado     348313 non-null  object 
dtypes: float64(4), int32(1), object(4)
memory usage: 25.2+ MB


None


--------------------------------  Estadísticas descriptivas de las columnas


,anno,area_sembrada_ha,area_cosechada_ha,produccion_t,rendimiento_t_ha
count,"348,313.00","348,313.00","348,313.00","348,313.00","345,260.00"
mean,"2,016.24",264.45,233.20,"2,934.98",9.81
std,5.33,"1,150.49","1,031.04","46,207.89",15.20
min,"2,006.00",0.00,0.00,0.00,0.00
25%,"2,012.00",8.00,6.00,27.20,1.60
50%,"2,017.00",28.41,24.00,124.70,5.67
75%,"2,021.00",125.00,107.00,600.95,12.00
max,"2,024.00","60,000.00","61,000.00","4,776,340.50",253.00


## Explorar para cuales cultivos existen series largas

In [15]:
# Extraer lista de cultivos disponibles
cultivos_disponibles = sorted(UPRA_2007_2024['cultivo_armonizado'].unique())

# Mostrar número de observaciones municipio-año disponibles de cada cultivo
conteo_registros = (
    UPRA_2007_2024.groupby(['cultivo_armonizado'])[columnas_numericas].count()
        .reset_index()
        .sort_values(by='area_sembrada_ha', ascending=False)
        .reset_index(drop=True)
)

suma_registros = (
    UPRA_2007_2024.groupby(['cultivo_armonizado'])[columnas_numericas].sum()
        .reset_index()
        .sort_values(by='area_sembrada_ha', ascending=False)
        .reset_index(drop=True)
)

# Mostrar resumen de registros
print(MSC_SEPARADOR, "Conteo de registros municipio-año por cultivo (20 primeros)")
display(conteo_registros.head(20))
suma_conteo_20primeros = conteo_registros.head(20)[columnas_numericas].sum().to_frame().transpose()
suma_conteo_todos = conteo_registros[columnas_numericas].sum().to_frame().transpose()
print("Los 20 primeros representan este porcentaje del total de registros")
display(suma_conteo_20primeros/suma_conteo_todos)

print(MSC_SEPARADOR, "Suma de magnitudes por cultivo (20 primeros)")
display(suma_registros.head(20))
suma_suma_20primeros = suma_registros.head(20)[columnas_numericas].sum().to_frame().transpose()
suma_suma_todos = suma_registros[columnas_numericas].sum().to_frame().transpose()
print("Los 20 primeros representan este porcentaje del total de registros")
display(suma_suma_20primeros/suma_suma_todos)
# Para rendimiento, el resultado de esta suma no tiene sentido económico. Solo se incluyó por el procesamiento de datos


-------------------------------- Conteo de registros municipio-año por cultivo (20 primeros)


,cultivo_armonizado,area_sembrada_ha,area_cosechada_ha,produccion_t,rendimiento_t_ha
0,Maíz,44383,44383,44383,44334
1,Frijol,23572,23572,23572,23544
2,Yuca,17996,17996,17996,17970
3,Tomate,16159,16159,16159,16146
4,Plátano,13875,13875,13875,13840
5,Papa,12565,12565,12565,12559
6,Caña,12106,12106,12106,12062
7,Arroz,11796,11796,11796,11786
8,Café,11071,11071,11071,11035
9,Arveja,10418,10418,10418,10417


Los 20 primeros representan este porcentaje del total de registros


,area_sembrada_ha,area_cosechada_ha,produccion_t,rendimiento_t_ha
0,0.67,0.67,0.67,0.67



-------------------------------- Suma de magnitudes por cultivo (20 primeros)


,cultivo_armonizado,area_sembrada_ha,area_cosechada_ha,produccion_t,rendimiento_t_ha
0,Café,"16,014,408.67","13,243,603.24","13,777,349.88","10,591.34"
1,Arroz,"10,919,929.20","10,538,874.04","55,514,666.38","47,780.64"
2,Maíz,"10,879,069.82","9,929,695.07","26,602,119.56","92,168.00"
3,Palma de aceite,"10,039,306.77","8,280,409.53","27,746,724.47","6,520.68"
4,Caña,"8,813,477.41","7,706,420.37","554,662,871.09","328,268.14"
5,Plátano,"8,090,460.17","7,304,411.01","67,250,703.16","104,052.78"
6,Yuca,"3,925,707.70","3,440,897.51","38,257,996.01","187,150.62"
7,Cacao,"3,737,506.47","3,080,839.26","1,690,960.48","5,165.48"
8,Papa,"3,335,073.71","3,145,661.95","63,296,949.28","198,263.06"
9,Frijol,"2,236,306.01","2,067,940.33","2,690,921.82","28,746.01"


Los 20 primeros representan este porcentaje del total de registros


,area_sembrada_ha,area_cosechada_ha,produccion_t,rendimiento_t_ha
0,0.93,0.93,0.92,0.41


## Extraer información de multiples cultivos
Para cada `cultivo_interes` en `lista_de_cultivos`, extraigo y organizo
1. Área Sembrada del cultivo de todos los demás cultivos (suma)
1. Área Cosechada del cultivo de todos los demás cultivos (suma)
1. Producción del cultivo de todos los demás cultivos (suma)

### Definir la lista de cultivos principales

In [16]:
# Calculo las variables de interés para los 20 cultivos con más registros (por conteo y por suma)
lista_de_cultivos_conteo = (
    conteo_registros
        .sort_values(by='area_sembrada_ha', ascending=False)
        .reset_index(drop=True)
        .loc[:20,'cultivo_armonizado']
        .to_list()
    )

lista_de_cultivos_suma = (
    suma_registros
        .sort_values(by='area_sembrada_ha', ascending=False)
        .reset_index(drop=True)
        .loc[:20,'cultivo_armonizado']
        .to_list()
    )

print(MSC_SEPARADOR, "20 cultivos con más observaciones de area sembrada:\n", lista_de_cultivos_conteo)
print(MSC_SEPARADOR, "20 cultivos con mayor magnitud de area sembrada:\n", lista_de_cultivos_suma)

# Encontrar la lista de los cultivos principales entre los de más registros y los de mayor area sembrada
lista_de_cultivos_principales = list(
    dict.fromkeys(lista_de_cultivos_suma + lista_de_cultivos_conteo)
)

print(MSC_SEPARADOR, f"Cultivos principales por número de registros y area sembrada. N = {len(lista_de_cultivos_principales)}\n",
      lista_de_cultivos_principales)


-------------------------------- 20 cultivos con más observaciones de area sembrada:
 ['Maíz', 'Frijol', 'Yuca', 'Tomate', 'Plátano', 'Papa', 'Caña', 'Arroz', 'Café', 'Arveja', 'Cacao', 'Aguacate', 'Habichuela', 'Limón', 'Ahuyama', 'Lulo', 'Mora', 'Naranja', 'Tomate de árbol', 'Patilla', 'Mango']

-------------------------------- 20 cultivos con mayor magnitud de area sembrada:
 ['Café', 'Arroz', 'Maíz', 'Palma de aceite', 'Caña', 'Plátano', 'Yuca', 'Cacao', 'Papa', 'Frijol', 'Banano', 'Aguacate', 'Soya', 'Ñame', 'Mango', 'Otros cítricos', 'Arveja', 'Naranja', 'Algodón', 'Piña', 'Coco']

-------------------------------- Cultivos principales por número de registros y area sembrada. N = 29
 ['Café', 'Arroz', 'Maíz', 'Palma de aceite', 'Caña', 'Plátano', 'Yuca', 'Cacao', 'Papa', 'Frijol', 'Banano', 'Aguacate', 'Soya', 'Ñame', 'Mango', 'Otros cítricos', 'Arveja', 'Naranja', 'Algodón', 'Piña', 'Coco', 'Tomate', 'Habichuela', 'Limón', 'Ahuyama', 'Lulo', 'Mora', 'Tomate de árbol', 'Patilla']

### Extraer información de los cultivos de interés

In [17]:
# Lista con el nombre de los cultivos de interés
cultivos_disponibles = lista_de_cultivos_principales.copy()

# Lista de las variables de interés
vbles_resultado = [COL_area_sembrada, COL_area_cosechada, COL_produccion] # NO incluyo rendimiento porque no se puede agregar igual que las otras columnas

# DataFrame vacio para almacenar resultados
panel_cultivos = pd.DataFrame()

for cultivo_interes in tqdm(cultivos_disponibles):
    print(MSC_SEPARADOR, " Procesando: ", cultivo_interes)
    
    # Extraer datos del cultivo de interés ---------------------------------------------------------------------
    filas_cultivo_interes = UPRA_2007_2024['cultivo_armonizado']==cultivo_interes
    UPRA_cultivo_interes = UPRA_2007_2024.loc[filas_cultivo_interes]

    # Dar estructura al DF
    UPRA_cultivo_interes = UPRA_cultivo_interes.melt(
        id_vars=[COL_ID_MUNICIPIO, COL_ANNO, 'cultivo_armonizado'],
        value_vars=vbles_resultado,
        var_name=COL_VARIABLE_MEDICION,
        value_name=COL_VALOR
    )

    # organizar información del Panel
    UPRA_cultivo_interes[COL_CLASIFICACION_ECONOMETRIA] = 'Resultado' # Identificar el tipo de variable en el planteamiento econometrico
    UPRA_cultivo_interes = UPRA_cultivo_interes.rename(columns={'cultivo_armonizado':COL_VARIABLE_SUJETO})
    UPRA_cultivo_interes[COL_VARIABLE_DETALLE] = ''
    UPRA_cultivo_interes[COL_NOMBRE_DE_VARIABLE] = ''
    UPRA_cultivo_interes[COL_VARIABLE_DESCRIPCION] = (UPRA_cultivo_interes[COL_VARIABLE_SUJETO] + ': ' +
                                                     UPRA_cultivo_interes[COL_VARIABLE_MEDICION] + ' de todos los tipos de ' + 
                                                     UPRA_cultivo_interes[COL_VARIABLE_SUJETO])

    # Extraer datos de los cultivos diferentes al cultivo de interés ------------------------------------------
    # Calcular la suma del Area sembrada, Area cosechada y Producción para los demás cultivos dentro del municipio-año

    # Organizar información de los demás cultivos por grupo_de_cultivo i.e 'grupo_armonizado' ---------------------------------
    UPRA_diferente_a_cultivo_interes_porgrupocultivo = UPRA_2007_2024.loc[~filas_cultivo_interes].groupby(
        [COL_ID_MUNICIPIO, COL_ANNO, 'grupo_armonizado', ])[vbles_resultado].sum()
    UPRA_diferente_a_cultivo_interes_porgrupocultivo = UPRA_diferente_a_cultivo_interes_porgrupocultivo.reset_index() 
    UPRA_diferente_a_cultivo_interes_porgrupocultivo[COL_CLASIFICACION_ECONOMETRIA] = 'Control' # Identificar el tipo de variable en el planteamiento econometrico

    # Dar estructura al DF
    UPRA_diferente_a_cultivo_interes_porgrupocultivo = UPRA_diferente_a_cultivo_interes_porgrupocultivo.melt(
        id_vars=[COL_ID_MUNICIPIO, COL_ANNO, 'grupo_armonizado'],
        value_vars=vbles_resultado,
        var_name=COL_VARIABLE_MEDICION,
        value_name=COL_VALOR
    )

    # organizar información del Panel
    UPRA_diferente_a_cultivo_interes_porgrupocultivo[COL_VARIABLE_SUJETO] = cultivo_interes
    UPRA_diferente_a_cultivo_interes_porgrupocultivo[COL_CLASIFICACION_ECONOMETRIA] = 'Control'
    UPRA_diferente_a_cultivo_interes_porgrupocultivo[COL_VARIABLE_DETALLE] = 'Grupo cultivo:' + UPRA_diferente_a_cultivo_interes_porgrupocultivo['grupo_armonizado']
    UPRA_diferente_a_cultivo_interes_porgrupocultivo[COL_NOMBRE_DE_VARIABLE] = ''
    UPRA_diferente_a_cultivo_interes_porgrupocultivo[COL_VARIABLE_DESCRIPCION] = (UPRA_diferente_a_cultivo_interes_porgrupocultivo[COL_VARIABLE_SUJETO] + ': ' +
                                                                    UPRA_diferente_a_cultivo_interes_porgrupocultivo[COL_VARIABLE_MEDICION] + ' de los demás cultivos - grupo de cultivos ' +
                                                                    UPRA_diferente_a_cultivo_interes_porgrupocultivo['grupo_armonizado'])


    # Extraer datos de los cultivos diferentes al cultivo de interés ------------------------------------------
    # Calcular la suma del Area sembrada, Area cosechada y Producción para los demás cultivos dentro del municipio-año

    # Organizar información de los demás cultivos por grupo_de_cultivo i.e 'ciclo_armonizado' ---------------------------------
    UPRA_diferente_a_cultivo_interes_porciclocultivo = UPRA_2007_2024.loc[~filas_cultivo_interes].groupby(
        [COL_ID_MUNICIPIO, COL_ANNO, 'ciclo_armonizado', ])[vbles_resultado].sum()
    UPRA_diferente_a_cultivo_interes_porciclocultivo = UPRA_diferente_a_cultivo_interes_porciclocultivo.reset_index()
    UPRA_diferente_a_cultivo_interes_porciclocultivo[COL_CLASIFICACION_ECONOMETRIA] = 'Control' # Identificar el tipo de variable en el planteamiento econometrico

    # Dar estructura al DF
    UPRA_diferente_a_cultivo_interes_porciclocultivo = UPRA_diferente_a_cultivo_interes_porciclocultivo.melt(
        id_vars=[COL_ID_MUNICIPIO, COL_ANNO, 'ciclo_armonizado'],
        value_vars=vbles_resultado,
        var_name=COL_VARIABLE_MEDICION,
        value_name=COL_VALOR
    )

    # organizar información del Panel
    UPRA_diferente_a_cultivo_interes_porciclocultivo[COL_VARIABLE_SUJETO] = cultivo_interes
    UPRA_diferente_a_cultivo_interes_porciclocultivo[COL_CLASIFICACION_ECONOMETRIA] = 'Control'
    UPRA_diferente_a_cultivo_interes_porciclocultivo[COL_VARIABLE_DETALLE] = 'Ciclo cultivo:' + UPRA_diferente_a_cultivo_interes_porciclocultivo['ciclo_armonizado']
    UPRA_diferente_a_cultivo_interes_porciclocultivo[COL_NOMBRE_DE_VARIABLE] = ''
    UPRA_diferente_a_cultivo_interes_porciclocultivo[COL_VARIABLE_DESCRIPCION] = (UPRA_diferente_a_cultivo_interes_porciclocultivo[COL_VARIABLE_SUJETO] + ': ' +
                                                                    UPRA_diferente_a_cultivo_interes_porciclocultivo[COL_VARIABLE_MEDICION] + ' de los demás cultivos - cultivos de ciclo ' +
                                                                    UPRA_diferente_a_cultivo_interes_porciclocultivo['ciclo_armonizado'])

    # Unir bases de datos
    panel_cultivos = pd.concat([
        panel_cultivos,
        UPRA_cultivo_interes[ORDEN_DF],
        UPRA_diferente_a_cultivo_interes_porgrupocultivo[ORDEN_DF],
        UPRA_diferente_a_cultivo_interes_porciclocultivo[ORDEN_DF]
    ])
    
#display(UPRA_cultivo_interes[ORDEN_DF].head(2))
#display(UPRA_diferente_a_cultivo_interes_porgrupocultivo[ORDEN_DF].head(2))
#display(UPRA_diferente_a_cultivo_interes_porciclocultivo[ORDEN_DF].head(2))

  0%|                                                                                           | 0/29 [00:00<?, ?it/s]


--------------------------------  Procesando:  Café


  3%|██▊                                                                                | 1/29 [00:00<00:17,  1.63it/s]


--------------------------------  Procesando:  Arroz


  7%|█████▋                                                                             | 2/29 [00:01<00:17,  1.56it/s]

...

 93%|████████████████████████████████████████████████████████████████████████████▎     | 27/29 [00:29<00:02,  1.49s/it]


--------------------------------  Procesando:  Tomate de árbol


 97%|███████████████████████████████████████████████████████████████████████████████▏  | 28/29 [00:31<00:01,  1.53s/it]


--------------------------------  Procesando:  Patilla


100%|██████████████████████████████████████████████████████████████████████████████████| 29/29 [00:32<00:00,  1.14s/it]


In [18]:
# Mostrar estructura del panel
print(MSC_SEPARADOR, "Estructura del panel final")
display(panel_cultivos)


-------------------------------- Estructura del panel final


,codigo_dane_municipio,anno,nombre_variable,variable_sujeto,variable_medicion,variable_detalle,variable_descripcion,valor,clasificacion_econometria
0,05101,2007,,Café,area_sembrada_ha,,Café: area_sembrada_ha de todos los tipos de Café,"9,680.00",Resultado
1,05034,2007,,Café,area_sembrada_ha,,Café: area_sembrada_ha de todos los tipos de Café,"8,696.00",Resultado
2,05642,2007,,Café,area_sembrada_ha,,Café: area_sembrada_ha de todos los tipos de Café,"6,011.00",Resultado
3,05209,2007,,Café,area_sembrada_ha,,Café: area_sembrada_ha de todos los tipos de Café,"5,578.00",Resultado
4,05091,2007,,Café,area_sembrada_ha,,Café: area_sembrada_ha de todos los tipos de Café,"5,439.40",Resultado
...,...,...,...,...,...,...,...,...,...
118486,99773,2022,,Patilla,produccion_t,Ciclo cultivo:Transitorio,Patilla: produccion_t de los demás cultivos - ...,"4,337.50",Control
118487,99773,2023,,Patilla,produccion_t,Ciclo cultivo:Permanente,Patilla: produccion_t de los demás cultivos - ...,"22,640.50",Control
118488,99773,2023,,Patilla,produccion_t,Ciclo cultivo:Transitorio,Patilla: produccion_t de los demás cultivos - ...,"4,375.20",Control
118489,99773,2024,,Patilla,produccion_t,Ciclo cultivo:Permanente,Patilla: produccion_t de los demás cultivos - ...,"23,536.00",Control


In [19]:
print(MSC_SEPARADOR, 'Mostrar mediciones disponibles')
display(panel_cultivos['variable_descripcion'].unique())


-------------------------------- Mostrar mediciones disponibles


array(['Café: area_sembrada_ha de todos los tipos de Café',
       'Café: area_cosechada_ha de todos los tipos de Café',
       'Café: produccion_t de todos los tipos de Café',
       'Café: area_sembrada_ha de los demás cultivos - grupo de cultivos Hortalizas',
       'Café: area_sembrada_ha de los demás cultivos - grupo de cultivos Leguminosas',
       'Café: area_sembrada_ha de los demás cultivos - grupo de cultivos Raíces y tubérculos',
       'Café: area_sembrada_ha de los demás cultivos - grupo de cultivos Cultivos tropicales tradicionales',
       'Café: area_sembrada_ha de los demás cultivos - grupo de cultivos Frutales',
       'Café: area_sembrada_ha de los demás cultivos - grupo de cultivos Cultivos para condimentos, bebidas medicinales y aromáticas',
       'Café: area_sembrada_ha de los demás cultivos - grupo de cultivos Cereales',
       'Café: area_sembrada_ha de los demás cultivos - grupo de cultivos Oleaginosas',
       'Café: area_cosechada_ha de los demás cultivos - 

# Exportar datos intermedios

### Exportar datos de diagnóstico

In [3]:
# Crear resumenes de datos para exportar
resumen_por_cultivo = panel_cultivos.groupby([COL_VARIABLE_SUJETO, COL_VARIABLE_MEDICION])[COL_VALOR].agg(['count', 'sum']).unstack()
resumen_por_anno = panel_cultivos.groupby([COL_ANNO, COL_VARIABLE_MEDICION])[COL_VALOR].agg(['count', 'sum']).unstack()

# Guardar diagnosticos
save_diagnostic(df=panel_cultivos,
                df_name = "panel_cultivos",
                filepath=DIAGNOSTICS / f"e1001_process_UPRA_panel_cultivos.md",
                additional_summary=[resumen_por_cultivo, resumen_por_anno]
               )

print(MSC_SEPARADOR, "Resumen por cultivo")
display(resumen_por_cultivo)

print(MSC_SEPARADOR, "Resumen por anno")
display(resumen_por_anno)

### Exportar Panel

In [205]:
panel_cultivos.to_parquet(DATA/'intermediate/e1001_panel_cultivos_UPRA.parquet')